Same Thing as spamclass But this time here we use the Word2vec using gensim
As There the splitting was incorrect that caused the testing data to get leaked 

In [36]:
import nltk
import pandas as pd
import numpy as np

Data Ingestion 

In [37]:
message = pd.read_csv("SMSSpamCollection.txt",
sep = "\t",
names = ["label","message"])

Perform Text Preprocessing 1 on entire data 

In [38]:
#These should be done on all data before split
import re #regular expression
#Stopwords
from nltk.corpus import stopwords 
stop_words = stopwords.words("english")
#Lemmatize
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

#Corpus Building
corpus = []
for i in range(len(message)):
    #Remove extra marks
    changes = re.sub("[^a-zA-Z0-9]"," ",message["message"][i])
    #Lowercase ALL
    changes = changes.lower()
    #Tokenize 
    changes = changes.split()

    #Lemmatize every word individually and remove stopwords
    changes = [lemmatizer.lemmatize(word) for word in changes if not word in stop_words and word !=""]
    
    #Rejoin the words back to full sentences
    changes= " ".join(changes)

    #Finally build the corpus
    corpus.append(changes)



Note All data and prepare target data 

In [39]:
X_text = corpus
y = pd.get_dummies(message["label"]).iloc[:,1].values

In [40]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Now apply tokenizations separately on both train and test data so that there is no data leak

In [41]:
from gensim.utils import simple_preprocess #Converts Document into a lower case tokens ignoreing tokens that are too shot or too long

train_words = [simple_preprocess(sent) for sent in X_train_text]

test_words = [simple_preprocess(sent) for sent in X_test_text]

Text Preprocessing 2 : Creating Custom  word2Vec Model

In [42]:
import gensim

In [43]:
#Train word2vec From scratch only on train data 
model = gensim.models.Word2Vec(
    train_words, #Source
    window = 5, #Window Size
    min_count = 2 ,
    vector_size=100,
    workers=4
                               )

In [44]:
#Gives us all the words that have been created using out custom word to vector
model.wv.index_to_key

['call',
 'get',
 'ur',
 'gt',
 'lt',
 'go',
 'day',
 'free',
 'ok',
 'know',
 'got',
 'come',
 'good',
 'time',
 'want',
 'love',
 'like',
 'text',
 'txt',
 'send',
 'today',
 'need',
 'da',
 'home',
 'one',
 'sorry',
 'going',
 'take',
 'lor',
 'stop',
 'still',
 'see',
 'reply',
 'dont',
 'back',
 'new',
 'mobile',
 'hi',
 'co',
 'think',
 'later',
 'week',
 'tell',
 'phone',
 'please',
 'pls',
 'message',
 'msg',
 'hope',
 'night',
 'make',
 'claim',
 'say',
 'dear',
 'thing',
 'happy',
 'great',
 'min',
 'hey',
 'friend',
 'wat',
 'yes',
 'work',
 'oh',
 'way',
 'www',
 'well',
 'give',
 'number',
 'much',
 'life',
 'amp',
 'right',
 'yeah',
 'cash',
 'already',
 'let',
 'win',
 'ask',
 'anything',
 'miss',
 'said',
 'tomorrow',
 'prize',
 'thanks',
 'im',
 'service',
 'uk',
 'babe',
 'would',
 'last',
 'find',
 'really',
 'care',
 'morning',
 'meet',
 'tone',
 'first',
 'com',
 'year',
 'also',
 'every',
 'urgent',
 'pick',
 'lol',
 'contact',
 'keep',
 'box',
 'nokia',
 'sure',


Impplement Average Word to Vec

In [45]:
def avg_w2v(doc):
    vector = [
        model.wv[word]
        for word in doc 
        if word in model.wv
    ]
    if len(vector) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vector,axis= 0)

Vectorize the train and testing data separately

In [46]:
X_train = np.array([avg_w2v(doc) for doc in train_words])
X_test = np.array([avg_w2v(doc) for doc in test_words])

Model Creation

1. Random Forest

In [47]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
classifier.fit(X_train,y_train)
y_pred = classifier.predict(X_test)

In [48]:
from sklearn.metrics import accuracy_score,classification_report

acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")

Model Accuracy : 0.9542600896860987
Model Classification Report :
               precision    recall  f1-score   support

       False       0.95      1.00      0.97       966
        True       0.98      0.67      0.80       149

    accuracy                           0.95      1115
   macro avg       0.97      0.83      0.89      1115
weighted avg       0.96      0.95      0.95      1115



2. Logistic Regression

In [49]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(X_train,y_train)
y_pred = classifier.predict(X_test)

In [50]:
acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")

Model Accuracy : 0.8663677130044843
Model Classification Report :
               precision    recall  f1-score   support

       False       0.87      1.00      0.93       966
        True       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

Using SVC instead of Logistic regression

In [51]:
from sklearn.svm import SVC
classifier = SVC()
classifier.fit(X_train,y_train)
y_pred = classifier.predict(X_test)

In [52]:
acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")

Model Accuracy : 0.8663677130044843
Model Classification Report :
               precision    recall  f1-score   support

       False       0.87      1.00      0.93       966
        True       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/anshuman/D_Drive/Coding/DataScience/anaconda3/envs/nlp/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 